<a href="https://colab.research.google.com/github/wintermute-HC/ar-model-uploader/blob/main/physical_ai_lecture_01_exercise_ipynb_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Physical AI 基礎編 知能ロボティクス講座 2026 Spring 第1回 演習資料
## LeRobot × SmolVLA × LIBERO で学ぶ VLA 推論までの流れ

このノートブックでは、Hugging Face の軽量な VLA モデル **SmolVLA** の LIBERO 用チェックポイント `HuggingFaceVLA/smolvla_libero` を使い、ロボットのマニピュレーション用ベンチマーク **LIBERO** 上で
**推論 → 評価 → 可視化** までを体験します。

## GPU の確認

まず Colab に **T4 GPU** が割り当てられているか確認します。
`nvidia-smi` の出力に `Tesla T4` と `15360MiB`（約 16GB）程度の VRAM が表示されれば OK です。

In [ ]:
!nvidia-smi

## 依存関係のインストール

LeRobot を **`smolvla`（SmolVLA 用）** と **`libero`（LIBERO 環境用）** の extra 付きでインストールします。

(セッションリスタートが必要になります)

In [ ]:
!apt-get update
!apt-get install -y \
    libegl1 libegl1-mesa-dev libgles2-mesa-dev \
    libglfw3 libosmesa6-dev libgl1-mesa-glx

!pip install "lerobot[smolvla,libero]==0.5.1"

## SmolVLA とは

**SmolVLA** は Hugging Face が 2025 年に公開した軽量な **Vision-Language-Action (VLA)** モデルです。「**1 枚の GPU で学習でき、コンシューマー GPU や CPU でも動く**」ことを目標に設計されています。

### 学習データ
- **LeRobot コミュニティが公開した 487 個の実機データセット**で事前学習
- 大企業の非公開・巨大データではなく、**コミュニティの公開データ中心**で作られている点が特徴（同種モデルの学習データより 1 桁以上小さい）

### 性能（LIBERO ベンチマーク, 論文報告値）
| モデル | パラメータ数 | 平均成功率 |
| --- | --- | --- |
| Octo | 0.09B | 75.1% |
| OpenVLA | 7B | 76.5% |
| π0 | 3.3B | 86.0% |
| **SmolVLA** | **0.45B** | **87.3%** |

本ノートブックで使う `HuggingFaceVLA/smolvla_libero` は、この SmolVLA を **LIBERO 向けに学習したチェックポイント**です。

参考:
- 論文 https://arxiv.org/abs/2506.01844
- 解説ブログ https://huggingface.co/blog/smolvla

## LIBERO とは

**LIBERO** はロボットポリシーの性能評価によく用いられる**マニピュレーションのためのベンチマーク**です（Liu et al., NeurIPS 2023）。
シミュレータの中でのマニピュレーション（物を掴む・置く・開ける等）タスクを定義しています。

### 構成
- 物理シミュレータ **MuJoCo** 上の **Franka Emika Panda** アーム
- **5 つのタスクスイート（合計 130 タスク）**。各タスクには人間のデモ（テレオペ）が付属

| スイート | タスク数 | 何を変えて評価するか |
| --- | --- | --- |
| `libero_spatial` | 10 | 物の配置（空間関係）の違い |
| `libero_object` | 10 | 対象物体の違い |
| `libero_goal` | 10 | ゴールの違い |
| `libero_10` (LIBERO-Long) | 10 | 長時間・複合タスク |
| `libero_90` | 90 | 多様な短タスク |

### 入出力
- **観測 (observation)**
  - `observation.state`: 8 次元の proprio（手先の位置・姿勢 6 + グリッパ（2指として扱う） 2）
  - `observation.images.image`: メインカメラ（俯瞰）
  - `observation.images.image2`: 手首カメラ
  - `task`: 自然言語の指示文（例: "pick up the black bowl and put it on the plate"）
- **アクション (action)**: 7 次元（手先の位置・姿勢の変位 6 + グリッパ開閉 1）

### 評価指標
- 各タスクを複数エピソードずつ rollout し、**タスク成功率 (success rate)** を測ります。
- 「成功」はタスクごとの達成条件（対象物が目標位置・状態になったか）で自動判定されます。

公式ドキュメント:
- https://huggingface.co/docs/lerobot/libero
- https://libero-project.github.io/main

## LeRobot とは

**LeRobot** は Hugging Face が開発する、**実機ロボット学習のためのオープンソース・フレームワーク**です。モデル等はすべて PyTorch で記述されています。
**データ・モデル・学習・評価・実機制御**を一気通貫で扱うことができ、現在様々なプロジェクトで LeRobot 形式のデータセットが用いられています。

### 主な構成要素
- **`LeRobotDataset`**: ロボットのエピソード（画像・proprio・アクション・言語指示）を統一フォーマットで扱うデータセット。
  HF Hub から共有・ダウンロードできる（本ノートブックで用いる `HuggingFaceVLA/libero` もこちら）
- **ロボットポリシー**: ACT, Diffusion Policy, π0, **SmolVLA** など各種ロボットポリシーの実装が含まれている
- **環境**: LIBERO, PushT などシミュレーション環境のラッパ

公式: https://github.com/huggingface/lerobot ／ ドキュメント: https://huggingface.co/docs/lerobot

## モデルのロード

チェックポイント **`HuggingFaceVLA/smolvla_libero`** を `SmolVLAPolicy.from_pretrained` でロードします（`transformers` ライブラリなどとインターフェースは類似している）

あわせて **前処理（preprocessor）／後処理（postprocessor）** パイプラインを `make_pre_post_processors` で構築します。
これらは推論時に必須で、次の役割を持ちます。

- **preprocessor**: 観測辞書のキー変換 → バッチ次元の付与 → 言語指示のトークナイズ → デバイス転送 → **正規化**
- **postprocessor**: モデル出力アクションの **逆正規化** → CPU への転送

正規化統計はチェックポイント内に保存されているため、`make_pre_post_processors` の **第 2 引数に
チェックポイント ID を渡す**ことで、学習時と同じ統計が読み込まれます（3 節で触れた「学習時と推論時で処理を一致させる」要）。

In [ ]:
import os
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

MODEL_ID = "HuggingFaceVLA/smolvla_libero"
device = "cuda"

# チェックポイントをロード（初回は数百 MB のダウンロードが走る）
model = SmolVLAPolicy.from_pretrained(MODEL_ID)
model = model.to(device)
model.eval()

# 前処理・後処理パイプライン（正規化統計はチェックポイントから読み込まれる）
preprocess, postprocess = make_pre_post_processors(
    model.config,
    MODEL_ID,
    preprocessor_overrides={"device_processor": {"device": device}},
)

torch.cuda.synchronize()
!nvidia-smi

## 推論デモ

LIBERO データセット **`HuggingFaceVLA/libero`** から **1 エピソードだけ**取り出し、その先頭フレームをモデルに入力して **アクションチャンク**を出力させます。

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# 1 エピソードだけロード
dataset = LeRobotDataset("HuggingFaceVLA/libero", episodes=[0])

# エピソード 0 の先頭フレーム index を取得
ep = 0
from_idx = dataset.meta.episodes["dataset_from_index"][ep]
frame = dataset[from_idx]

observation = {
    "observation.state": frame["observation.state"],
    "observation.images.image": frame["observation.images.image"],
    "observation.images.image2": frame["observation.images.image2"],
    "task": frame["task"],
}

print("=== 入力（観測）の shape ===")
print("state       :", tuple(observation["observation.state"].shape))
print("image       :", tuple(observation["observation.images.image"].shape), "(C, H, W)")
print("image2      :", tuple(observation["observation.images.image2"].shape), "(C, H, W)")
print("task (指示文):", repr(observation["task"]))

入力画像（メインカメラと手首カメラ）を表示して、モデルが「何を見ているか」を確認します。

In [ ]:
import matplotlib.pyplot as plt

# データセットの画像は (C, H, W) なので表示用に (H, W, C) へ permute
img_main = observation["observation.images.image"].permute(1, 2, 0).cpu().numpy()
img_wrist = observation["observation.images.image2"].permute(1, 2, 0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_main); axes[0].set_title("agentview (main)"); axes[0].axis("off")
axes[1].imshow(img_wrist); axes[1].set_title("eye-in-hand (wrist)"); axes[1].axis("off")
plt.suptitle(observation["task"])
plt.tight_layout()
plt.show()

前処理 → 推論 → 後処理の流れを実行します。`predict_action_chunk` で **アクションチャンク全体**を、
`select_action` で **1 ステップ分のアクション**を取得して、それぞれの shape を確認します。

In [ ]:
model.reset()  # 各エピソードの開始時にアクションキューをクリア
processed = preprocess(observation)                  # 前処理（バッチ次元付与・トークナイズ・正規化など）
action_chunk = model.predict_action_chunk(processed) # (B, n_action_steps, action_dim)
print("=== 出力（アクションチャンク） ===")
print("action_chunk:", tuple(action_chunk.shape), "= (バッチ, ステップ数, アクション次元)")
print("アクション値の例:", [round(x, 3) for x in action_chunk[0][0].tolist()])
print("postprocess後:", [round(x, 3) for x in postprocess(action_chunk)[0][0].tolist()])


- `action_chunk` の `(B, n_action_steps, 7)` は、「1 回の推論で `n_action_steps` ステップ先までの
  アクションをまとめて予測している」ことを表します。これは VLA の **action chunking** と呼ばれます。
- 実際の制御では、この chunk を内部キューに溜めて `select_action` が 1 ステップずつ取り出し、キューが空になったら再推論します。

## ベンチマーク評価

LeRobot では、本来は `lerobot-eval` コマンドを利用することで、LIBERO シミュレータ上に実際に rollout して **タスク成功率**を測ることができます。
ただ、今回は `lerobot-eval` を用いると中身で何をやっているかわからないため、評価スクリプトを書いて内容を理解します。

In [ ]:
EVAL_OUT = "./eval_out"

# 以下をコメントアウトすることで lerobot-eval を実行させることも可能
# cmd = (
#     "lerobot-eval"
#     f" --policy.path={MODEL_ID}"
#     " --policy.device=cuda"
#     " --policy.n_action_steps=10"
#     " --env.type=libero"
#     " --env.task=libero_spatial"
#     ' --env.task_ids="[0]"'
#     " --eval.n_episodes=3"
#     " --eval.batch_size=1"
#     " --env.max_parallel_tasks=1"
#     f" --output_dir={EVAL_OUT}"
# )
# print(cmd, "\n")
# !{cmd}

In [ ]:
import time
import numpy as np
import torch
from libero.libero import benchmark
from lerobot.envs.libero import LiberoEnv
import imageio

# ===== 評価設定 =====
SUITE_NAME = "libero_spatial"   # 評価するスイート
TASK_ID = 0                     # スイート内のタスク番号
N_EPISODES = 3                  # 評価エピソード数

# action chunk
model.config.n_action_steps = 10

# ===== 環境を作る =====
suite = benchmark.get_benchmark_dict()[SUITE_NAME]()
env = LiberoEnv(
    task_suite=suite,
    task_id=TASK_ID,
    task_suite_name=SUITE_NAME,
    obs_type="pixels_agent_pos",
    control_mode="relative",
    observation_width=256,
    observation_height=256,
    init_states=True,
    n_envs=1,
)
max_steps = env._max_episode_steps
print(f"タスク指示: {env.task_description}")
print(f"1 エピソードの最大ステップ: {max_steps}\n")


def quat2axisangle(quat):  # quat: torch (4,) 順序は (x, y, z, w)
    w = quat[3].clamp(-1.0, 1.0)
    den = torch.sqrt(torch.clamp(1.0 - w * w, min=0.0))
    if den < 1e-10:
        return torch.zeros(3, dtype=torch.float32)
    return (quat[:3] / den * (2.0 * torch.acos(w))).float()

def to_chw(img_hwc):
    img = np.ascontiguousarray(img_hwc[::-1, ::-1])
    return torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

def build_observation(obs):
    rs = obs["robot_state"]
    eef_pos = torch.as_tensor(rs["eef"]["pos"], dtype=torch.float32)       # (3,)
    eef_quat = torch.as_tensor(rs["eef"]["quat"], dtype=torch.float32)     # (4,) (x,y,z,w)
    gripper = torch.as_tensor(rs["gripper"]["qpos"], dtype=torch.float32)  # (2,)
    state = torch.cat([eef_pos, quat2axisangle(eef_quat), gripper])        # (8,)
    return {
        "observation.state": state,
        "observation.images.image": to_chw(obs["pixels"]["image"]),
        "observation.images.image2": to_chw(obs["pixels"]["image2"]),
        "task": env.task_description,
    }


results = []         # 各エピソードの成功 / 失敗
rollout_frames = []  # 動画用フレーム
t0 = time.time()

for ep in range(N_EPISODES):
    env.init_state_id = ep   # 初期状態を固定（lerobot と同じく 0,1,2,... を使う）
    obs, info = env.reset()
    model.reset()            # アクションチャンクのキューをクリア
    success = False

    for t in range(max_steps):
        print(f"Episode {ep}: step {t + 1}")

        rollout_frames.append(np.ascontiguousarray(obs["pixels"]["image"][::-1, ::-1]))

        # 前処理 → 1 ステップ分のアクションを取得 → 後処理（逆正規化）
        processed = preprocess(build_observation(obs))
        with torch.inference_mode():
            action = model.select_action(processed)
        action = postprocess(action).squeeze(0).cpu().numpy()

        obs, reward, terminated, truncated, info = env.step(action)
        success = success or bool(info.get("is_success", False))
        if terminated:
            break

    results.append(success)
    print(f"  Episode {ep}: {'✅ 成功' if success else '❌ 失敗'}  ({t + 1} steps)")

    # Save the video
    VIDEO_PATH = f"rollout_ep{ep}.mp4"
    imageio.mimsave(VIDEO_PATH, rollout_frames, fps=30)
    rollout_frames = []

env.close()

success_rate = 100.0 * sum(results) / len(results)
print("\n=== 評価結果 ===")
print(f"成功率: {success_rate:.1f}%  ({sum(results)}/{len(results)} episodes)")
print(f"所要時間: {time.time() - t0:.1f} 秒")

In [ ]:
from IPython.display import Video, display

for ep in range(N_EPISODES):
    VIDEO_PATH = f"rollout_ep{ep}.mp4"
    print(f"エピソード {ep}: {VIDEO_PATH}")
    display(Video(VIDEO_PATH, embed=True, width=480))